# Adding custom components to SUPREME

SUPREME can be installed as a dependency (`pip install supreme-unlearning`) and extended with custom datasets, models, unlearning methods, baselines, and metrics, without modifying framework code.

The framework is registry-based. A component is resolved by name through a naming convention (`Foo` -> `supreme.<subpackage>.Foo`). A custom component is added by registering a module path that points at external code. Resolution order is: runtime overrides, then packaging entry points, then the built-in convention. Custom components therefore neither collide with nor alter the built-ins. The component interfaces and the Lightning Fabric integration rules are documented in [`docs/extending.md`](../docs/extending.md).

## 1. Install

The PyPI distribution name is `supreme-unlearning`; the import name is `supreme`.

```bash
pip install supreme-unlearning            # core (CPU / MPS)
pip install "supreme-unlearning[cuda]"    # adds DeepSpeed, bitsandbytes, NVIDIA telemetry
```

`import supreme` is lightweight: it does not import the PyTorch/Lightning stack, so registration works before the heavy dependencies are loaded.

In [ ]:
# Optional: install SUPREME if it isn't already in this kernel's environment.
# Use the %pip magic (not !pip) so it installs into THIS notebook's kernel, not
# a different Python on PATH. On Linux + NVIDIA use "supreme-unlearning[cuda]".
# Skips quickly if the requirement is already satisfied.
%pip install supreme-unlearning

In [2]:
import supreme

print("SUPREME", supreme.__version__)
print("register API:", [n for n in supreme.__all__ if n.startswith("register")])

SUPREME 0.1.2.dev0
register API: ['register_model', 'register_baseline', 'register_unlearning_method', 'register_metric', 'register_dataset']


## 2. Registry check without PyTorch (no GPU required)

Registration and resolution do not depend on PyTorch, so this section runs in any environment. It defines a small external module, registers one method and one model from it, and confirms that SUPREME resolves the names to that module path and can load them. Real components are PyTorch-based and are covered in section 3.

In [3]:
import importlib


# Registry check without torch. Define two trivial components here (no torch),
# register the objects directly, and confirm the framework resolves and calls them.
def acme_method(fabric=None, model=None, **kwargs):
    # A real method returns the unlearned model; this stub returns a marker to
    # demonstrate resolution and invocation without the heavy stack.
    return "acme_method ran"


def AcmeNet(num_labels=10, **kwargs):
    # A real model factory returns an nn.Module; a dict here keeps it torch-free.
    return {"kind": "AcmeNet", "num_labels": num_labels}


supreme.register_unlearning_method(
    "acme_method", acme_method
)  # pass the object, not a path
supreme.register_model("AcmeNet", AcmeNet)

from supreme import registry as R

print(
    "method ->", R.resolve_method_location("acme_method")
)  # ('__main__', 'acme_method')
print("model  ->", R.resolve_callable_location("model", "AcmeNet"))


def _load(loc):
    mod, attr = loc
    return getattr(importlib.import_module(mod), attr)


print("call method:", _load(R.resolve_method_location("acme_method"))())
print(
    "call model :", _load(R.resolve_callable_location("model", "AcmeNet"))(num_labels=7)
)

# Registration also syncs the framework's name lists, so the CLI and validation accept them:
print("in all_methods :", "acme_method" in supreme.project_config.all_methods)
print("in model_names :", "AcmeNet" in supreme.project_config.model_names)

method -> ('__main__', 'acme_method')
model  -> ('__main__', 'AcmeNet')
call method: acme_method ran
call model : {'kind': 'AcmeNet', 'num_labels': 7}
in all_methods : True
in model_names : True


## 3. Define the components

Each component follows the interface documented in [`docs/extending.md`](../docs/extending.md): an unlearning method, an evaluation metric, a model factory, and a dataset class. The components below are defined directly in notebook cells. Section 4 registers the objects themselves, so no files or module paths are needed; `register_*` accepts a live callable or class and resolves it in the current process. In a packaged project these would live in normal modules of the package (the entry-points route in section 5). The components are PyTorch-based and require the full SUPREME stack.

### 3a. Unlearning method

The framework always passes `fabric`, `model`, `model_name`, `num_gpus`, `wandb_logging_flag`, `type_of_unlearning_strategy`, and the standard retain/forget train and test dataloaders. Each model is paired with its optimiser through `fabric.setup(...)`, and gradients are synchronised with `fabric.backward(loss)`.

In [4]:
import torch
import torch.nn as nn


# An unlearning method receives the Fabric handle, the model to unlearn (an
# independent copy of the original) and the retain/forget dataloaders, plus num_gpus,
# wandb_logging_flag and more (accept **kwargs). It MUST RETURN the unlearned model -
# the framework captures it (compare src/supreme/methods/unlearning_methods/finetune.py).
def gentle_finetune(
    fabric,
    model,
    retain_train_dataloader,
    num_gpus=1,
    wandb_logging_flag=False,
    lr=0.01,
    epochs=1,
    **kwargs,
):
    """Toy method: fine-tune on the retain set so the model forgets the rest."""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    # The framework already ran fabric.setup(model); unwrap to the raw module then
    # re-setup it WITH the optimizer (mirrors finetune/ssd/scrub).
    raw_model = model.module if hasattr(model, "module") else model
    model, optimizer = fabric.setup(raw_model, optimizer)
    model.train()
    for _ in range(epochs):
        for images, _, labels in retain_train_dataloader:  # datasets yield (x, _, y)
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(images), labels)
            fabric.backward(loss)  # NOT loss.backward() - Fabric syncs grads
            optimizer.step()
    return model  # <-- the framework uses the returned (unlearned) model

### 3b. Evaluation metric

A metric returns a dict of named results. The `@track_evaluation_metric` decorator wraps that dict in the standard result envelope (`{"metric_value_dict": ..., "core_time_dict": ..., ...}`). Setting `track_evaluation_resources=True` (default `False`) additionally profiles the metric's own time, memory, and utilisation, which serves to compare metrics by cost rather than to compare unlearning methods. Results are aggregated across processes with `fabric.all_gather(...)`. Metric names not recognised by the built-in dispatcher are resolved through the registry and run automatically, with no edits to `metrics_main.py`. A metric that needs the retrained reference model is registered with `requires_retrain=True`.

In [5]:
import torch
from supreme.utils.unlearning.evaluation_utils import track_evaluation_metric


# A metric returns a DICT of named results. @track_evaluation_metric wraps that in
# the standard envelope; with track_evaluation_resources=True (default False) it also
# profiles the metric's OWN time/memory/compute-utilisation (for comparing the metrics
# by cost). It receives fabric, num_gpus, the model and the retain/forget/test
# dataloaders - use the external-dispatch names (e.g. unlearned_model,
# forget_test_dataloader); compare src/supreme/eval_metrics/membership_inference_attack.py.
@track_evaluation_metric
def forget_accuracy(
    fabric, num_gpus, unlearned_model, forget_test_dataloader, **kwargs
):
    """Accuracy on the forget set (lower is better after unlearning)."""
    unlearned_model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, _, labels in forget_test_dataloader:
            preds = unlearned_model(images).argmax(dim=1)
            correct += (preds == labels).sum()
            total += labels.numel()
    acc = correct.float() / max(int(total), 1)
    acc = fabric.all_gather(acc).mean().item()
    return {"forget_accuracy": acc}

### 3c. Model and dataset

A model is a factory function returning an `nn.Module`. A dataset is a class taking the framework's constructor kwargs (`root`, `download`, `train`, `unlearning`, `img_size`, `model_name`); subclassing an existing builder in [`src/supreme/datasets/datasets.py`](../src/supreme/datasets/datasets.py) is the simplest route. The dataset is registered with its data `root`, the `class_dict` (class name -> integer label) used by class and subclass unlearning, and optionally the per-architecture training schedule.

In [6]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder


# Model factory: takes num_labels (+ kwargs) and returns an nn.Module - same contract
# as src/supreme/models/ResNet18.py (def ResNet18(num_labels)) and ViT.py.
def TinyCNN(num_labels, **kwargs):
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(16, num_labels),
    )


# Dataset: subclass a torchvision dataset, take (root, train, unlearning, download,
# img_size, model_name), build the transforms, and have __getitem__ return a 3-TUPLE
# (x, <placeholder>, y) - matches the framework's loader unpacking (`for x, _, y in ...`).
# Compare PinsFaceRecognition / Cifar10 in src/supreme/datasets/datasets.py.
class MyFaces(ImageFolder):
    def __init__(self, root, train, unlearning, download, img_size=32, model_name=None):
        transform = transforms.Compose(
            [
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
            ]
        )
        super().__init__(root, transform)

    def __getitem__(self, index):
        x, y = super().__getitem__(index)
        return x, torch.Tensor([]), y

/path/to/supreme-unlearning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Register the components (runtime API)

The calls below run once, before the pipeline is launched (for example at the top of a run script).

In [7]:
import supreme

# Register the components defined above by passing the OBJECTS directly - no module
# paths, no files. This works because they live in this kernel's __main__ (see the
# section-6 note on in-process vs the grid driver). A dataset accepts a live class too.
supreme.register_unlearning_method("gentle_finetune", gentle_finetune)
supreme.register_metric("forget_accuracy", forget_accuracy, requires_retrain=False)
supreme.register_model("TinyCNN", TinyCNN)
supreme.register_dataset(
    "MyFaces",
    MyFaces,  # a live class, accepted directly
    root="/data/myfaces",
    class_dict={"alice": 0, "bob": 1, "carol": 2},
    rn_epochs=100,
    rn_milestones=[30, 60, 80],  # ResNet schedule
    vit_epochs=8,
    vit_milestones=[7],  # ViT schedule
)

pc = supreme.project_config
print("method  registered:", "gentle_finetune" in pc.all_methods)
print("metric  registered:", "forget_accuracy" in pc.evaluation_metrics)
print("model   registered:", "TinyCNN" in pc.model_names)
print("dataset registered:", "MyFaces" in pc.dataset_names)

method  registered: True
metric  registered: True
model   registered: True
dataset registered: True


## 5. Alternative: ship a plugin via packaging entry points

Instead of calling `register_*` at runtime, an installed package can declare its components in its packaging metadata, which SUPREME discovers on first use. The callable categories use the direct `module:attribute` groups:

```toml
# in the plugin package's pyproject.toml
[project.entry-points."supreme.unlearning_methods"]
gentle_finetune = "my_extras_method:gentle_finetune"

[project.entry-points."supreme.metrics"]
forget_accuracy = "my_extras_metric:forget_accuracy"

[project.entry-points."supreme.models"]
TinyCNN = "my_extras_model:TinyCNN"
```

Datasets carry extra metadata (root, class dict, schedule), and several components can be registered at once. For these cases the `supreme.plugins` group points at a zero-argument setup callable that registers them through the runtime API:

```toml
[project.entry-points."supreme.plugins"]
my_extras = "my_extras_register:setup"
```

The `supreme.plugins` setup callable lives in a normal module of the package, for example `my_extras/register.py`:

```python
import supreme
from my_extras.method import gentle_finetune
from my_extras.dataset import MyFaces

def setup():
    """Discovered via the 'supreme.plugins' entry-point group on first use."""
    supreme.register_unlearning_method("gentle_finetune", gentle_finetune)
    supreme.register_dataset("MyFaces", MyFaces, root="/data/myfaces",
                             class_dict={"alice": 0, "bob": 1, "carol": 2})
```

Runtime registration (sections 3 and 4) applies only to the current process. An installed plugin is importable in any process, so it also works with the `run_local.sh` and SLURM drivers, which spawn subprocesses.

## 6. Run the pipeline with the registered components

Registered names are passed to `run_training` / `run_unlearning` (or the `supreme-train` / `supreme-unlearn` console scripts) exactly like the built-in names. The cell below runs the full pipeline (train -> unlearn -> evaluate) in-process, using the custom model, method, and metric on the built-in Cifar10 dataset (auto-downloaded). A custom dataset such as `MyFaces` works the same way once its data is in place.

In-process registration versus the grid driver: runtime `register_*` calls apply to the current process, so they reach in-process `run_training` / `run_unlearning`. The `run_local.sh` driver spawns fresh subprocesses that do not see runtime registration; for that route, components are shipped as an installed plugin via packaging entry points (section 5).

This run is small: it trains on MPS/CPU, logs to W&B in offline mode (so no data reaches the cloud account), and uses one epoch. It checks the end-to-end wiring, not the paper's accuracy.

In [8]:
# Full pipeline (train -> unlearn -> evaluate) with the registered components,
# in-process so the runtime register_* calls above apply. Uses built-in Cifar10
# (auto-downloads) so it is self-contained. Small by design (epochs=1).
import os, glob, multiprocessing as mp

mp.set_start_method("fork", force=True)  # macOS DataLoader workers (notebook-safe)
os.environ.update(
    {
        "WANDB_MODE": "offline",
        "TRAINING_SEED": "260",
        "UNLEARNING_SEED": "260",
        "WANDB_PROJECT_PREFIX": "DEMO",
    }
)

import supreme

pc = supreme.project_config
pc.Cifar10_RN_EPOCHS = 1  # one epoch (real default is 20); checks wiring
pc.Cifar10_RN_MILESTONES = []

# 1) train the custom model
supreme.run_training(
    [
        "-unlearning_seed",
        "260",
        "-precision",
        "32-true",
        "-net",
        "TinyCNN",
        "-dataset",
        "Cifar10",
        "-classes",
        "10",
        "-unlearning_context",
        "random",
        "-include_gpus_in_path",
        "true",
        "-training_seed",
        "260",
    ]
)

# 2) locate the checkpoint just written (the driver does this via update_checkpoint_paths.py)
ckpt = sorted(
    glob.glob(
        os.path.join(
            pc.CHECKPOINT_PATH,
            "precision_32-true/1gpus/no_dist/train_seed_260/unlearning_seed_260/"
            "model_checkpoints/TinyCNN/Cifar10/*/*-best.pth",
        )
    )
)[-1]

# 3) per-cell unlearning output dir (run_local.sh exports this as LOG_DIR)
log_dir = os.path.join(
    pc.PROJECT_ROOT,
    "logs/unlearning/precision_32-true/1gpus/"
    "train_seed_260/unlearn_seed_260/random_/Cifar10/TinyCNN/classes_10/forget_perc_0.01",
)
os.makedirs(log_dir, exist_ok=True)
os.environ["LOG_DIR"] = log_dir

# 4) unlearn (gentle_finetune), then evaluate (accuracy + the custom forget_accuracy).
#    unlearn_main runs in two env-toggled phases via PERFORM_EVALUATION.
args = [
    "-net",
    "TinyCNN",
    "-dataset",
    "Cifar10",
    "-type_of_unlearning_strategy",
    "random_",
    "-weight_path",
    ckpt,
    "-seed",
    "260",
    "-precision",
    "32-true",
    "-eval_metrics",
    "accuracy,forget_accuracy",
    "-classes",
    "10",
    "-forget_perc",
    "0.01",
]
supreme.run_unlearning(["-method", "gentle_finetune", *args])  # unlearning stage
os.environ["PERFORM_EVALUATION"] = "true"
supreme.run_unlearning(["-method", "gentle_finetune", *args])  # evaluation stage
print(
    "Done: trained TinyCNN, unlearned with gentle_finetune, evaluated accuracy + forget_accuracy."
)

Seed set to 260



################################################################################
### STARTING TRAINING PHASE ###
### Model: TinyCNN | Dataset: Cifar10 | Classes: 10 ###
### Precision: 32-true | Seed: 260 ###
### World size: 1 | Local devices: 1 | Rank: 0 ###
### Distributed strategy: ddp ###
################################################################################
Using training seed 260 for reproducible training
Loading file: 'TinyCNN' from module: '__main__'
[PARAM COUNT] model=TinyCNN num_labels=10 total=618 (0.00M) initialised model with 618 parameters
No weights loaded
File /path/to/supreme-unlearning/logs/training/precision_32-true/processed_datasets/seed_260/Cifar10/TinyCNN/trainset.pt does not exist. Saving it.


Files already downloaded and verified


File /path/to/supreme-unlearning/logs/training/precision_32-true/processed_datasets/seed_260/Cifar10/TinyCNN/testset.pt does not exist. Saving it.
Files already downloaded and verified


Checkpoint path: /path/to/supreme-unlearning/logs/training/precision_32-true/1gpus/no_dist/train_seed_260/unlearning_seed_260/model_checkpoints/TinyCNN/Cifar10/Tuesday_02_June_2026_21h_40m_00s
Checkpoint file template: /path/to/supreme-unlearning/logs/training/precision_32-true/1gpus/no_dist/train_seed_260/unlearning_seed_260/model_checkpoints/TinyCNN/Cifar10/Tuesday_02_June_2026_21h_40m_00s/{model}-{dataset}-{epoch}-{type}.pth

=== Starting Epoch 1/1 ===
Starting training phase...


...

[W ParallelNative.cpp:230] Warning: Cannot set number of intraop threads after parallel work has started or after set_num_threads call when using native parallel backend (function set_num_threads)


[W ParallelNative.cpp:230] Warning: Cannot set number of intraop threads after parallel work has started or after set_num_threads call when using native parallel backend (function set_num_threads)


====== METRICS SUMMARY ======
eval_result: {'accuracy': {'test': {'unlearning_method': {'whole_acc': {'final_value': 31.002769470214844}, 'retain_acc': {'final_value': 31.002769470214844}, 'forget_acc': {'final_value': 32.623924255371094}, 'core_time_elapsed': {'final_value': 1.0852410793304443}, 'memory_usage': {'total_gpu_memory': 0.009319782257080078, 'total_cpu_memory': 0.0210418701171875, 'max_gpu_memory': 0.009319782257080078, 'max_cpu_memory': 0.0210418701171875}, 'compute_utilisation': {'gpu_ids': '0', 'start_compute_util': {'total': 0.0, 'max': 0.0}, 'end_compute_util': {'total': 3.0, 'max': 3.0}, 'total_avg_compute_util': 3.0999999046325684, 'total_peak_compute_util': 26.0, 'max_avg_compute_util': 3.0999999046325684, 'max_peak_compute_util': 26.0, 'total_compute_seconds': 3.3642473220825195, 'total_compute_hours': 0.0009345131693407893, 'logical_cpu_count': 16, 'total_avg_cpu_util': 3.8848958015441895, 'total_peak_cpu_util': 6.712500095367432, 'max_avg_cpu_util': 3.8848958015

Garbage collection complete
Cleared MPS cache and synchronized
✓ Cleanup completed successfully - safe to exit
Done: trained TinyCNN, unlearned with gentle_finetune, evaluated accuracy + forget_accuracy.


[W ParallelNative.cpp:230] Warning: Cannot set number of intraop threads after parallel work has started or after set_num_threads call when using native parallel backend (function set_num_threads)
[W ParallelNative.cpp:230] Warning: Cannot set number of intraop threads after parallel work has started or after set_num_threads call when using native parallel backend (function set_num_threads)
